# 05 · Merge, join, concat

Joining two tables is where row counts silently change. Every section starts with two
tiny tables you can check by eye, then (sometimes) the same thing on the real files.

**What's in here**
- `merge` with `how=` inner / left / right / outer
- key cardinality: `validate=`, and the row-multiplication pitfall
- `indicator=True` to find orphans
- different key names, merging on the index, several keys, `suffixes`
- key dtype mismatches, tz-aware vs tz-naive keys
- `concat` rows and columns
- `merge_asof`: join to the latest available record (forecasts, prices)
- `combine_first` / `update`: patch gaps from a second source
- merge checklist

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 1. Two small tables

`left` is a list of meters, `right` is a list of readings. The key is `meter_id`.
Notice: `M3` is only in `left`, `M9` is only in `right`.

In [2]:
left = pd.DataFrame({"meter_id": ["M1", "M2", "M3"], "region": ["North", "South", "North"]})
right = pd.DataFrame({"meter_id": ["M1", "M2", "M9"], "kwh": [10, 20, 90]})
print(left)
print()
print(right)

  meter_id region
0       M1  North
1       M2  South
2       M3  North

  meter_id  kwh
0       M1   10
1       M2   20
2       M9   90


## 2. `how=` decides which keys survive

`inner`: only keys present in **both**.

In [3]:
pd.merge(left, right, on="meter_id", how="inner")

,meter_id,region,kwh
0,M1,North,10
1,M2,South,20


`M3` and `M9` are gone. Two rows in, two rows out — but a row was silently dropped from each side.

`left`: every row of `left`, NaN where `right` has nothing.

In [4]:
pd.merge(left, right, on="meter_id", how="left")

,meter_id,region,kwh
0,M1,North,10.0
1,M2,South,20.0
2,M3,North,NaN


`M3` is kept with `kwh = NaN`. Note `kwh` became float because of the NaN.

`outer`: every key from either side.

In [5]:
pd.merge(left, right, on="meter_id", how="outer")

,meter_id,region,kwh
0,M1,North,10.0
1,M2,South,20.0
2,M3,North,NaN
3,M9,NaN,90.0


`right` is the mirror image of `left` (every row of `right`, NaN for `M9`'s region).

In [6]:
pd.merge(left, right, on="meter_id", how="right")

,meter_id,region,kwh
0,M1,North,10
1,M2,South,20
2,M9,NaN,90


## 3. `indicator=True`: where did each row come from?

Adds a `_merge` column: `both`, `left_only`, `right_only`. Use it with `how="outer"` to find orphans.

In [7]:
both = pd.merge(left, right, on="meter_id", how="outer", indicator=True)
both

,meter_id,region,kwh,_merge
0,M1,North,10.0,both
1,M2,South,20.0,both
2,M3,North,NaN,left_only
3,M9,NaN,90.0,right_only


In [8]:
both[both["_merge"] != "both"]        # the orphans

,meter_id,region,kwh,_merge
2,M3,North,NaN,left_only
3,M9,NaN,90.0,right_only


## 4. Row counts: always check before and after

The single most useful habit: print `len()` of both inputs and of the result.

In [9]:
merged = pd.merge(left, right, on="meter_id", how="inner")
print("left  :", len(left))
print("right :", len(right))
print("result:", len(merged))

left  : 3
right : 3
result: 2


## **Pitfall:** a duplicated key on one side multiplies rows

`right2` has `M1` twice (two readings). Each `left` row for `M1` matches **both**, so the
result has one row per match.

In [10]:
right2 = pd.DataFrame({"meter_id": ["M1", "M1", "M2"], "kwh": [10, 11, 20]})
print(right2)

  meter_id  kwh
0       M1   10
1       M1   11
2       M2   20


In [11]:
m = pd.merge(left, right2, on="meter_id", how="left")
print("left rows  :", len(left))
print("result rows:", len(m))
m

left rows  : 3
result rows: 4


,meter_id,region,kwh
0,M1,North,10.0
1,M1,North,11.0
2,M2,South,20.0
3,M3,North,NaN


3 rows in, 4 rows out. With a many-to-many key (duplicates on **both** sides) the count
explodes: 2 × 2 = 4 rows for that key alone.

In [12]:
left2 = pd.DataFrame({"meter_id": ["M1", "M1"], "tariff": ["Fixed", "TOU"]})
pd.merge(left2, right2, on="meter_id")

,meter_id,tariff,kwh
0,M1,Fixed,10
1,M1,Fixed,11
2,M1,TOU,10
3,M1,TOU,11


## 5. `validate=`: make pandas check the cardinality you assumed

If you *believe* the key is unique on the right, say so. pandas raises if it is not.

In [13]:
try:
    pd.merge(left, right2, on="meter_id", how="left", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", e)

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge


**Interview check:** *"What would happen to your row count if a meter appeared twice in the
dimension table?"* — Every reading of that meter is duplicated; totals double for it.
`validate="one_to_many"` (left unique) or `"many_to_one"` (right unique) catches it.

In [14]:
ok = pd.merge(left, right, on="meter_id", how="left", validate="one_to_one")
print("passed, rows:", len(ok))

passed, rows: 3


## 6. Different key names: `left_on` / `right_on`

Both key columns are kept in the result.

In [15]:
readings = pd.DataFrame({"mpan": ["M1", "M2"], "kwh": [10, 20]})
pd.merge(left, readings, left_on="meter_id", right_on="mpan")

,meter_id,region,mpan,kwh
0,M1,North,M1,10
1,M2,South,M2,20


## 7. Merging on the index: `join`

`join` matches the index of the left frame to the index (or a column) of the right.

In [16]:
attrs = pd.DataFrame({"region": ["North", "South"]}, index=["M1", "M2"])
usage = pd.DataFrame({"kwh": [10, 20, 30]}, index=["M1", "M2", "M3"])
print(attrs)
print()
print(usage)

   region
M1  North
M2  South

    kwh
M1   10
M2   20
M3   30


In [17]:
usage.join(attrs)                 # left join by default

,kwh,region
M1,10,North
M2,20,South
M3,30,NaN


## 8. Several keys

Pass a list to `on=`. A row matches only if **all** keys agree.

In [18]:
a = pd.DataFrame({"meter_id": ["M1", "M1", "M2"], "month": [1, 2, 1], "kwh": [10, 12, 20]})
b = pd.DataFrame({"meter_id": ["M1", "M1", "M2"], "month": [1, 2, 2], "bill": [5, 6, 7]})
print(a)
print()
print(b)

  meter_id  month  kwh
0       M1      1   10
1       M1      2   12
2       M2      1   20

  meter_id  month  bill
0       M1      1     5
1       M1      2     6
2       M2      2     7


In [19]:
pd.merge(a, b, on=["meter_id", "month"], how="outer")

,meter_id,month,kwh,bill
0,M1,1,10.0,5.0
1,M1,2,12.0,6.0
2,M2,1,20.0,NaN
3,M2,2,NaN,7.0


`(M2, 1)` exists only in `a`, `(M2, 2)` only in `b` → NaN on the missing side.

## 9. Overlapping column names: `suffixes`

If both frames have a non-key column with the same name, pandas adds `_x` / `_y`. Name them yourself.

In [20]:
est = pd.DataFrame({"meter_id": ["M1", "M2"], "kwh": [9, 22]})
act = pd.DataFrame({"meter_id": ["M1", "M2"], "kwh": [10, 20]})
pd.merge(est, act, on="meter_id", suffixes=("_est", "_act"))

,meter_id,kwh_est,kwh_act
0,M1,9,10
1,M2,22,20


## **Pitfall:** key dtype mismatches

A string key on one side and an integer key on the other never match. pandas 2.x raises
for int vs str; for datetime vs string it also raises. Look at `dtypes` first.

In [21]:
x = pd.DataFrame({"id": [1, 2], "v": [10, 20]})
y = pd.DataFrame({"id": ["1", "2"], "w": [7, 8]})
print(x.dtypes["id"], "vs", y.dtypes["id"])
try:
    pd.merge(x, y, on="id")
except ValueError as e:
    print("ValueError:", str(e)[:80])

int64 vs object
ValueError: You are trying to merge on int64 and object columns for key 'id'. If you wish to


In [22]:
y["id"] = y["id"].astype(int)     # fix: make the dtypes agree
pd.merge(x, y, on="id")

,id,v,w
0,1,10,7
1,2,20,8


Datetime vs string: same story. Parse first.

In [23]:
p = pd.DataFrame({"time": pd.to_datetime(["2023-01-01", "2023-01-02"]), "v": [1, 2]})
q = pd.DataFrame({"time": ["2023-01-01", "2023-01-02"], "w": [7, 8]})
try:
    pd.merge(p, q, on="time")
except ValueError as e:
    print("ValueError:", str(e)[:80])
q["time"] = pd.to_datetime(q["time"])
pd.merge(p, q, on="time")

ValueError: You are trying to merge on datetime64[ns] and object columns for key 'time'. If 


,time,v,w
0,2023-01-01,1,7
1,2023-01-02,2,8


## **Pitfall:** tz-aware vs tz-naive keys

`2023-01-01 00:00+00:00` (aware) and `2023-01-01 00:00` (naive) are different things to
pandas. Merging them raises.

In [24]:
aware = pd.DataFrame({"time": pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00"], utc=True), "v": [1, 2]})
naive = pd.DataFrame({"time": pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00"]), "w": [7, 8]})
print(aware["time"].dtype)
print(naive["time"].dtype)
try:
    pd.merge(aware, naive, on="time")
except ValueError as e:
    print("ValueError:", str(e)[:70])

datetime64[ns, UTC]
datetime64[ns]
ValueError: You are trying to merge on datetime64[ns, UTC] and datetime64[ns] colu


In [25]:
naive["time"] = naive["time"].dt.tz_localize("UTC")     # say what the naive times mean
pd.merge(aware, naive, on="time")

,time,v,w
0,2023-01-01 00:00:00+00:00,1,7
1,2023-01-01 01:00:00+00:00,2,8


## 10. `concat`: stacking rows (axis=0)

Rows are appended. The index is kept as-is (so labels can repeat) unless `ignore_index=True`.

In [26]:
jan = pd.DataFrame({"meter_id": ["M1", "M2"], "kwh": [10, 20]})
feb = pd.DataFrame({"meter_id": ["M1", "M2"], "kwh": [11, 21]})
pd.concat([jan, feb])

,meter_id,kwh
0,M1,10
1,M2,20
0,M1,11
1,M2,21


In [27]:
pd.concat([jan, feb], ignore_index=True)

,meter_id,kwh
0,M1,10
1,M2,20
2,M1,11
3,M2,21


`keys=` labels which piece each row came from.

In [28]:
pd.concat([jan, feb], keys=["jan", "feb"])

meter_id  kwh
jan 0       M1   10
    1       M2   20
feb 0       M1   11
    1       M2   21

If the column sets differ, the missing cells become NaN.

In [29]:
mar = pd.DataFrame({"meter_id": ["M1"], "cost": [5.0]})
pd.concat([jan, mar], ignore_index=True)

,meter_id,kwh,cost
0,M1,10.0,NaN
1,M2,20.0,NaN
2,M1,NaN,5.0


## 11. `concat` side by side (axis=1) aligns on the index

Columns are placed next to each other; rows are matched **by index label**.

In [30]:
c1 = pd.DataFrame({"kwh": [10, 20]}, index=["M1", "M2"])
c2 = pd.DataFrame({"cost": [5, 9]}, index=["M2", "M3"])
pd.concat([c1, c2], axis=1)

,kwh,cost
M1,10.0,NaN
M2,20.0,5.0
M3,NaN,9.0


`M1` has no cost, `M3` has no kwh → NaN. Same alignment rule as Series arithmetic.

## 12. `merge_asof`: join to the latest record at or before

For each left row, take the right row with the largest key **≤** the left key
(`direction="backward"`, the default). Both sides must be sorted by the key.

In [31]:
obs = pd.DataFrame({
    "time": pd.to_datetime(["2023-01-01 10:00", "2023-01-01 10:20", "2023-01-01 10:40", "2023-01-01 11:00"]),
    "load": [1, 2, 3, 4],
})
fc = pd.DataFrame({
    "time": pd.to_datetime(["2023-01-01 09:30", "2023-01-01 10:30", "2023-01-01 11:30"]),
    "forecast": [100, 200, 300],
})
print(obs)
print()
print(fc)

                 time  load
0 2023-01-01 10:00:00     1
1 2023-01-01 10:20:00     2
2 2023-01-01 10:40:00     3
3 2023-01-01 11:00:00     4

                 time  forecast
0 2023-01-01 09:30:00       100
1 2023-01-01 10:30:00       200
2 2023-01-01 11:30:00       300


In [32]:
pd.merge_asof(obs, fc, on="time", direction="backward")

,time,load,forecast
0,2023-01-01 10:00:00,1,100
1,2023-01-01 10:20:00,2,100
2,2023-01-01 10:40:00,3,200
3,2023-01-01 11:00:00,4,200


- 10:00 and 10:20 → latest forecast at or before is 09:30 → 100
- 10:40 and 11:00 → 10:30 → 200
- 11:30 is never used (nothing on the left is at or after it)

`direction="forward"` takes the first record at or **after** instead.

In [33]:
pd.merge_asof(obs, fc, on="time", direction="forward")

,time,load,forecast
0,2023-01-01 10:00:00,1,200
1,2023-01-01 10:20:00,2,200
2,2023-01-01 10:40:00,3,300
3,2023-01-01 11:00:00,4,300


`tolerance` limits how far back to look; beyond it you get NaN.

In [34]:
pd.merge_asof(obs, fc, on="time", tolerance=pd.Timedelta("15min"))

,time,load,forecast
0,2023-01-01 10:00:00,1,NaN
1,2023-01-01 10:20:00,2,NaN
2,2023-01-01 10:40:00,3,200.0
3,2023-01-01 11:00:00,4,NaN


Unsorted input raises: sort first.

In [35]:
try:
    pd.merge_asof(obs.iloc[::-1], fc, on="time")
except ValueError as e:
    print("ValueError:", e)

ValueError: left keys must be sorted


`by=` does the as-of join within groups (one meter's readings only see that meter's forecasts).

In [36]:
obs2 = pd.DataFrame({"meter": ["A", "A", "B"], "time": pd.to_datetime(["2023-01-01 10:00", "2023-01-01 11:00", "2023-01-01 10:00"]), "load": [1, 2, 3]})
fc2 = pd.DataFrame({"meter": ["A", "B"], "time": pd.to_datetime(["2023-01-01 09:00", "2023-01-01 09:30"]), "forecast": [100, 300]})
pd.merge_asof(obs2.sort_values("time"), fc2.sort_values("time"), on="time", by="meter")

,meter,time,load,forecast
0,A,2023-01-01 10:00:00,1,100
1,B,2023-01-01 10:00:00,3,300
2,A,2023-01-01 11:00:00,2,100


**Interview check:** *"Why is `merge_asof` safer than a plain merge on the timestamp for
forecast data?"* — A plain merge on the target time takes **every** forecast for that hour,
including ones issued after your decision. As-of on the issue time only takes what existed.

### On real data: which forecast was available at 12:00 the day before?

`weather_forecasts.csv` has several forecasts per target hour (issued at different
`origin_datetime`). We want, for each target hour of day D, the forecast issued at or
before 12:00 on D−1.

In [37]:
fc = pd.read_csv("../data/weather_forecasts.csv", parse_dates=["origin_datetime", "forecast_datetime"])
fc[fc["forecast_datetime"] == "2023-01-10 16:00+00:00"]

,origin_datetime,forecast_datetime,horizon_h,temp_forecast_c
35847,2023-01-09 00:00:00+00:00,2023-01-10 16:00:00+00:00,40,1.52
35883,2023-01-09 12:00:00+00:00,2023-01-10 16:00:00+00:00,28,-5.38
35919,2023-01-10 00:00:00+00:00,2023-01-10 16:00:00+00:00,16,1.10
35955,2023-01-10 12:00:00+00:00,2023-01-10 16:00:00+00:00,4,-0.06


Four forecasts for the same hour. A plain merge would keep all four (4× the rows).
Step 1: the decision time for each target hour is 12:00 on the previous day.

In [38]:
hourly = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
target = hourly[["time"]].copy()
target["decision_time"] = (target["time"] - pd.Timedelta(days=1)).dt.floor("D") + pd.Timedelta(hours=12)
target.iloc[[30, 40, 50]]

,time,decision_time
30,2022-01-02 06:00:00+00:00,2022-01-01 12:00:00+00:00
40,2022-01-02 16:00:00+00:00,2022-01-01 12:00:00+00:00
50,2022-01-03 02:00:00+00:00,2022-01-02 12:00:00+00:00


Step 2: keep only forecasts issued at or before the decision time, for the right target
hour. That is a merge on `forecast_datetime` (exact) plus as-of on `origin_datetime`.

In [39]:
fc_sorted = fc.sort_values("origin_datetime")
target_sorted = target.sort_values("decision_time")
joined = pd.merge_asof(target_sorted, fc_sorted,
                       left_on="decision_time", right_on="origin_datetime",
                       left_by="time", right_by="forecast_datetime",
                       direction="backward")
joined = joined.sort_values("time").reset_index(drop=True)
joined[["time", "decision_time", "origin_datetime", "horizon_h", "temp_forecast_c"]].iloc[[30, 40, 50]]

,time,decision_time,origin_datetime,horizon_h,temp_forecast_c
30,2022-01-02 06:00:00+00:00,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,18.0,1.85
40,2022-01-02 16:00:00+00:00,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,28.0,7.50
50,2022-01-03 02:00:00+00:00,2022-01-02 12:00:00+00:00,2022-01-02 12:00:00+00:00,14.0,0.93


In [40]:
print("rows before:", len(target), " after:", len(joined))
print("horizons used:", joined["horizon_h"].min(), "to", joined["horizon_h"].max())

rows before: 17520  after: 17520
horizons used: 12.0 to 35.0


Same number of rows (no multiplication), and the horizons are 12–35 hours, exactly what a
12:00 D−1 decision implies. If you see horizon 1 here, you leaked.

## 13. `combine_first` / `update`: patch gaps from a second source

`combine_first`: where `primary` is NaN, take `backup`.

In [41]:
primary = pd.Series([1.0, np.nan, 3.0], index=["a", "b", "c"])
backup = pd.Series([9.0, 2.0, 9.0], index=["a", "b", "c"])
primary.combine_first(backup)

a    1.0
b    2.0
c    3.0
dtype: float64

`update` is in place and overwrites with every non-NaN value from the other side.

In [42]:
p = primary.copy()
p.update(pd.Series({"a": 100.0}))
p

a    100.0
b      NaN
c      3.0
dtype: float64

## Merge checklist

1. Print `len()` of both inputs; print `len()` of the result. Explain any change.
2. Is the key unique where you think it is? `df["key"].is_unique`, or `validate=`.
3. Same dtype on both sides? Same timezone?
4. Which `how`? Are dropped rows acceptable? Use `indicator=True` to see them.
5. For time-stamped data that arrives over time: `merge_asof` on the *issue* time.
6. Duplicated column names → `suffixes`.
7. After a left join, count NaN in the new columns: they are the unmatched rows.